In [1]:
#Import Libraries
import pandas as pd
import numpy as np
import os

In [2]:
path='/Users/Pc-Neumann/Desktop/Instacart Basket Analysis/02 Data/Prepared Data/df_ords_prods_merge.pkl'

In [3]:
df_prods_ords_merge=pd.read_pickle(os.path.join(path))

In [4]:
df = df_prods_ords_merge

In [5]:
df.shape

(1000000, 25)

In [6]:
df.head()

,Unnamed: 0_x,order_id,user_id,order_number,orders_day_of_week,order_time,days_since_prior_order,product_id,add_to_cart_order,reordered,...,prices,merge_indicator,price_range,price_range_loc,busiest_day,busiest_days,order_time_loc,time_of_order,max_order,loyalty_flag
0,0,2539329,1,1,2,8,NaN,196,1,0,...,9.0,both,Mid-range product,Mid-range product,Regularly busy,Regularly busy,Average Orders,Average Orders,10,New customer
1,0,2539329,1,1,2,8,NaN,14084,2,0,...,12.5,both,Mid-range product,Mid-range product,Regularly busy,Regularly busy,Average Orders,Average Orders,10,New customer
2,0,2539329,1,1,2,8,NaN,12427,3,0,...,4.4,both,Low-range product,Low-range product,Regularly busy,Regularly busy,Average Orders,Average Orders,10,New customer
3,0,2539329,1,1,2,8,NaN,26088,4,0,...,4.7,both,Low-range product,Low-range product,Regularly busy,Regularly busy,Average Orders,Average Orders,10,New customer
4,0,2539329,1,1,2,8,NaN,26405,5,0,...,1.0,both,Low-range product,Low-range product,Regularly busy,Regularly busy,Average Orders,Average Orders,10,New customer


In [7]:
path='/Users/Pc-Neumann/Desktop/Instacart Basket Analysis/02 Data/Prepared Data/df_ords_prods_merge1.pkl'

In [8]:
df_prods_ords_merge=pd.read_pickle(os.path.join(path))

In [30]:
df_prods_ords_merge=df

In [32]:
df.shape

(1000000, 25)

In [61]:
#repeat steps from Exercise

In [33]:
df.loc[df['max_order'] > 40, 'loyalty_flag'] = 'Loyal customer'

In [35]:
df.loc[(df['max_order'] <= 40) & (df['max_order'] > 10), 'loyalty_flag'] = 'Regular customer'

In [36]:
df.loc[df['max_order'] <= 10, 'loyalty_flag'] = 'New customer'

In [37]:
df['loyalty_flag'].value_counts(dropna=False)

loyalty_flag
Regular customer    482391
Loyal customer      322011
New customer        195598
Name: count, dtype: int64

In [39]:
#Task

The marketing team at Instacart wants to know whether there’s a difference between the spending habits of the three types of customers you identified. Use the loyalty flag you created and check the basic statistics of the product prices for each loyalty category (Loyal Customer, Regular Customer, and New Customer). What you’re trying to determine is whether the prices of products purchased by loyal customers differ from those purchased by regular or new customers.

In [38]:
df[['prices','user_id', 'loyalty_flag', 'order_number']].head(60)

,prices,user_id,loyalty_flag,order_number
0,9.0,1,New customer,1
1,12.5,1,New customer,1
2,4.4,1,New customer,1
3,4.7,1,New customer,1
4,1.0,1,New customer,1
5,9.0,1,New customer,2
6,3.0,1,New customer,2
7,4.4,1,New customer,2
8,10.3,1,New customer,2
9,4.7,1,New customer,2


In [40]:
df.groupby('loyalty_flag').agg({'prices': ['mean', 'min', 'max']})

prices              
                       mean  min      max
loyalty_flag                             
Loyal customer     9.081626  1.0  99999.0
New customer      15.843383  1.0  99999.0
Regular customer  11.139256  1.0  99999.0

In [41]:
#Answer

The minimum and maximum for each customer category don't differ from each other (I am considering too
try and exclude the 9999.0$ item)
The mean though differs. loyal customers and regular customers spend just 43% and 29% less than a new customer .  
Whereas a loyal customer only spends 20% less on the average than a regular customer

In [42]:
#Task

The team now wants to target different types of spenders in their marketing campaigns. This can be achieved by looking at the prices of the items people are buying. Create a spending flag for each user based on the average price across all their orders using the following criteria:

If the mean of the prices of products purchased by a user is lower than 10, then flag them as a “Low spender.”

If the mean of the prices of products purchased by a user is higher than or equal to 10, then flag them as a “High spender.”

In [62]:
#aggregate and groupby

In [46]:
df['spender_mean'] = df.groupby(['user_id'])['prices'].transform(np.mean)

/var/folders/2g/9y11dkqx34sfytlbfh8kk9dc0000gn/T/ipykernel_99930/1453802368.py:1: FutureWarning: The provided callable <function mean at 0x105ce98a0> is currently using SeriesGroupBy.mean. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "mean" instead.
  df['spender_mean'] = df.groupby(['user_id'])['prices'].transform(np.mean)


In [63]:
#define flag column

In [47]:
df.loc[df['spender_mean'] >= 10, 'spending_flag'] = 'High Spender'

/var/folders/2g/9y11dkqx34sfytlbfh8kk9dc0000gn/T/ipykernel_99930/869846210.py:1: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'High Spender' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[df['spender_mean'] >= 10, 'spending_flag'] = 'High Spender'


In [48]:
df.loc[df['spender_mean'] < 10, 'spending_flag'] = 'Low Spender'

In [64]:
#check outcome

In [49]:
df['spending_flag'].value_counts(dropna=False)

spending_flag
Low Spender     978614
High Spender     21386
Name: count, dtype: int64

In [50]:
#Task

In order to send relevant notifications to users within the app (for instance, asking users if they want to buy the same item again), the Instacart team wants you to determine frequent versus non-frequent customers. Create an order frequency flag that marks the regularity of a user’s ordering behavior according to the median in the “days_since_prior_order” column. The criteria for the flag should be as follows:

If the median of “days_since_prior_order” is higher than 20, then the customer should be labeled a “Non-frequent customer.”

If the median is higher than 10 and lower than or equal to 20, then the customer should be labeled a “Regular customer.”

If the median is lower than or equal to 10, then the customer should be labeled a “Frequent customer.”

In [51]:
df['frequency_median'] = df.groupby(['user_id'])['days_since_prior_order'].transform(np.median)

/var/folders/2g/9y11dkqx34sfytlbfh8kk9dc0000gn/T/ipykernel_99930/3481298194.py:1: FutureWarning: The provided callable <function median at 0x106244ea0> is currently using SeriesGroupBy.median. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "median" instead.
  df['frequency_median'] = df.groupby(['user_id'])['days_since_prior_order'].transform(np.median)


In [58]:
df.loc[df['frequency_median'] > 20, 'frequency_flag'] = 'Non-frequent customer'

In [53]:
df.loc[(df['frequency_median'] <= 40) & (df['frequency_median'] > 10), 'frequency_flag'] = 'Regular customer'

In [54]:
df.loc[df['frequency_median'] <= 10, 'frequency_flag'] = 'Frequent customer'

In [59]:
df['frequency_flag'].value_counts(dropna=False)

frequency_flag
Frequent customer        664849
Regular customer         219649
Non-frequent customer    115502
Name: count, dtype: int64

In [60]:
df.to_pickle(os.path.join(path,'/Users/Pc-Neumann/Desktop/Instacart Basket Analysis/02 Data/Prepared Data/df_prods_ords_frequencies.pkl'))

In [65]:
#export pickle